# 02 · B1 experiment (optional): multilingual sentence encoder + logistic regression

Compares a frozen multilingual encoder with B0 on the **same** held-out split. **Not a blocker** for B0 or the inference pipeline.

> **All data in this notebook is SYNTHETIC** (template-generated). Metrics are a regression/ablation signal on unseen template families — **not** an estimate of real-world accuracy. Hindi/Marathi text needs native-speaker review.

**Settings:** Accelerator = GPU (T4/P100) recommended; **Internet = ON** (downloads the encoder from Hugging Face on first run). Encoder availability could not be verified from the dev sandbox — if the default ID fails, set `ENCODER` to another multilingual encoder, or attach one as a Kaggle Model/Dataset and point `ENCODER` at its path.

*There is no production loader for B1.* If B1 clearly beats B0, exporting it (ONNX) and adding an inference loader is a follow-up decision.

In [ ]:
# --- 0. Locate the CivicConnect repo and set up a work directory (works on Kaggle and locally) ---------------
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def find_repo() -> Path:
    """Return a writable directory that contains ai/inference/schemas.py."""
    marker = Path("ai/inference/schemas.py")
    if os.environ.get("CIVIC_REPO"):                                   # explicit override
        return Path(os.environ["CIVIC_REPO"])
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:                                    # running from inside the repo
        if (p / marker).exists():
            return p
    if ON_KAGGLE:                                                      # attached Kaggle Dataset containing the repo or just ai/
        for hit in sorted(Path("/kaggle/input").glob("**/ai/inference/schemas.py"))[:1]:
            src = hit.parents[2]                                       # .../ai/inference/schemas.py -> parent of ai/
            dst = Path("/kaggle/working/repo")
            if not dst.exists():
                shutil.copytree(src, dst)                              # /kaggle/input is read-only
            return dst
        url = os.environ.get("CIVIC_GIT_URL")                          # optional: clone (needs Internet ON; use a Kaggle Secret for tokens)
        if url:
            subprocess.run(["git", "clone", "--depth", "1", url, "/kaggle/working/repo"], check=True)
            return Path("/kaggle/working/repo")
    raise SystemExit("Cannot find the repo. On Kaggle: add a Dataset containing the repo's `ai/` folder (see ai/training/README.md).")

REPO = find_repo()
sys.path.insert(0, str(REPO))
WORK = Path(os.environ.get("CIVIC_WORKDIR") or ("/kaggle/working" if ON_KAGGLE else tempfile.mkdtemp(prefix="civic_work_")))
WORK.mkdir(parents=True, exist_ok=True)

_MODULE = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers"}

def ensure(*pkgs):
    """pip install only what is missing (Kaggle images already ship numpy/scipy/scikit-learn/pydantic/httpx)."""
    missing = [p for p in pkgs if importlib.util.find_spec(_MODULE.get(p, p)) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ensure("numpy", "scipy", "scikit-learn", "pydantic", "httpx")
print(f"repo : {REPO}\nwork : {WORK}\npython {sys.version.split()[0]} | kaggle={ON_KAGGLE}")

In [ ]:
# ---- parameters ----
ENCODER = os.environ.get("CIVIC_ENCODER", "intfloat/multilingual-e5-small")   # alternatives: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
PREFIX = os.environ.get("CIVIC_ENCODER_PREFIX", "query: ")                    # e5 models expect "query: "; use "" for MiniLM
USE_FAKE_ENCODER = os.environ.get("CIVIC_B1_FAKE") == "1"                     # CI hook ONLY: exercises this notebook without torch/internet

if not USE_FAKE_ENCODER:
    ensure("sentence-transformers")
    import torch
    print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (slow but works)")

In [ ]:
from ai.training.src import build_dataset, train_encoder_head

DATA = WORK / "synthetic_v1"
if not (DATA / "train.jsonl").exists():
    build_dataset.main(["--out", str(DATA)])

embed_fn = None
if USE_FAKE_ENCODER:                                     # hashed n-gram stand-in; NOT a real encoder
    import numpy as np
    from ai.inference.providers.fake import FakeProvider
    _fp = FakeProvider(dim=256)
    def embed_fn(texts):
        return np.array([_fp._vec(t) for t in texts], dtype=np.float32)
    ENCODER = "fake/hashed-ngrams"

OUT = train_encoder_head.run(DATA, WORK / "b1_experiments", ENCODER, PREFIX, embed_fn=embed_fn)

## Compare with B0 (same held-out test families)

In [ ]:
import json

b1 = json.loads((OUT / "b1_report.json").read_text(encoding="utf-8"))["results"]
b0_path = next((p for p in [WORK / "reports" / "intake_b0_local.json", REPO / "ai/evaluation/reports/intake_b0_local.json"] if p.exists()), None)
b0 = json.loads(b0_path.read_text(encoding="utf-8"))["results"] if b0_path else None
print(f"{'':28s}{'subcat acc':>12s}{'category acc':>14s}{'macro-F1':>10s}")
if b0:
    print(f"{'B0 (tfidf+logreg)':28s}{b0['subcategory_accuracy']:>12}{b0['category_accuracy']:>14}{b0['subcategory_macro_f1']:>10}")
print(f"{'B1 (' + ENCODER[:20] + ')':28s}{b1['test_subcategory_accuracy']:>12}{b1['test_category_accuracy']:>14}{b1['test_macro_f1']:>10}")
print("\nB1 by language:", b1["test_subcategory_accuracy_by_language"])
print("\nNOTE: single held-out family per class/language -> wide intervals; do not over-read small differences.")

In [ ]:
import zipfile

BUNDLE = WORK / "civic_b1_report.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(OUT / "b1_report.json", f"b1_experiments/{OUT.name}/b1_report.json")
print("download:", BUNDLE)